# Debug Tracer — per-iteration visualization

`DebugTracer` is a toggleable debugging component for `iterative_serial`.
It records convergence history and optionally shows a 3-panel snapshot
(before Jdet / after Jdet / |Δφ|) after every N outer iterations.

Pass `debug=True` for default settings, or `debug=DebugTracer(show_every=2)` to customize.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dvfopt import iterative_serial, jacobian_det2D
from testcases import make_deformation, SYNTHETIC_CASES
from dvfopt.viz import DebugTracer, plot_grid_before_after

## 1. Build a test deformation field

Use a synthetic case that has negative Jacobian determinants.

In [ ]:
case_key = "01a_10x10_crossing"
dvf = make_deformation(case_key)

jac_init = jacobian_det2D(dvf[1:, ...])
print(f"Shape: {dvf.shape}")
print(f"Neg Jdet pixels: {int((jac_init <= 0).sum())}")
print(f"Min Jdet: {float(jac_init.min()):.4f}")

## 2. Run with debug tracing — show every iteration

Each outer iteration shows:
- **Left**: Jdet before the fix, with the optimisation window outlined
- **Middle**: Jdet after the fix, with negative-region contour
- **Right**: Displacement change magnitude |Δφ| — reveals exactly which region was modified

In [ ]:
tracer = DebugTracer(show_every=1, show_sub_iters=False)

phi = iterative_serial(
    dvf,
    verbose=1,
    debug=tracer,
)

## 3. Convergence history

In [ ]:
tracer.plot_history()

## 4. Before / after deformation grid

In [ ]:
phi_4d = np.stack([np.zeros_like(phi[0]), phi[0], phi[1]])[: , np.newaxis]
plot_grid_before_after(dvf, phi)

## 5. Try a harder case — show every 2 iterations, with sub-iteration tracing

In [ ]:
case_key2 = "01d_20x40_crossing"
dvf2 = make_deformation(case_key2)

jac2 = jacobian_det2D(dvf2[1:, ...])
print(f"Shape: {dvf2.shape}")
print(f"Neg Jdet pixels: {int((jac2 <= 0).sum())}")
print(f"Min Jdet: {float(jac2.min()):.4f}")

In [ ]:
tracer2 = DebugTracer(
    show_every=2,        # snapshot every 2 outer iterations
    show_sub_iters=True, # also show sub-iteration Jdet heatmaps
)

phi2 = iterative_serial(
    dvf2,
    verbose=1,
    debug=tracer2,
)

In [ ]:
tracer2.plot_history()

## 6. Silent tracing — no snapshots during run, only history afterwards

Set `show_every=0` to suppress inline plots while still collecting history.

In [ ]:
tracer3 = DebugTracer(show_every=0)  # silent

phi3 = iterative_serial(dvf, verbose=0, debug=tracer3)

# Inspect raw history arrays
print("Iterations:", tracer3.iterations)
print("Neg counts: ", tracer3.neg_counts)
print("Min Jdets:  ", [f"{v:.4f}" for v in tracer3.min_jdets])
print("Windows:    ", tracer3.window_sizes)
print("Sub-iters:  ", tracer3.sub_iter_counts)

In [ ]:
tracer3.plot_history()